In [1]:
import numpy as np
import pandas as pd
import os

# Organizando um erro que fizemos no Teste 4 nas Redes Bayesianas

No Teste 4, a coluna "Preciptacao recente" no "df_rede" foi só um renomeamento da classe qualitativa de preciptacao, a coluna "classe_precipitacao" no "df_final_limpo", mas calculo original não leva em consideração datas, como está no grafo da rede bayesiana, apenas a quantidade em mm. Ou seja, como cada linha da tabela (nesse caso do dataframe) repete os dias e muda principalmente as coordenadas, o cálculo qualitativo de cada precipitação está sendo feito para cada coordenada, que é um erro! Fizemos as predições se baseando nas coordenadas quando na verdade deveríamos ter feito com base na precipitação por Mês.

Abaixo terá o dataframe corrigido levando isso em consideração.


In [2]:
# dados brutos
# copiei de rebeca
pasta_1 = os.path.join("..", "DATA", "CSV_regioes") #caminho da pasta CSV_regioes
pasta_2 = os.path.join("..", "DATA", "CSV_acudes")  #caminho da pasta CSV_acudes

def ler_pasta(pasta, sep=";", **kwargs):
    lista = []
    arquivos = sorted(os.listdir(pasta))

    for i in arquivos:
        if i.lower().endswith(".csv"):
            caminho = os.path.join(pasta, i)

            df = pd.read_csv(caminho, sep=sep, **kwargs)
            df["arquivo"] = os.path.splitext(i)[0]
            lista.append(df)

    if not lista:
        print(f" n tem nada em: {pasta}. ")

    return lista

lista_dfs_precip = ler_pasta(pasta_1, sep=";", decimal=".") #le a pasta 1
lista_dfs_acude = ler_pasta(pasta_2, sep=";", decimal=",")  #le a pasta 2

# DataFrames de precipitação
if lista_dfs_precip:
    df_1 = pd.concat(lista_dfs_precip, ignore_index=True)
    display(df_1)

# DataFrames de açudes
if lista_dfs_acude:
    df_2 = pd.concat(lista_dfs_acude, ignore_index=True)
    df_2 = df_2.rename(columns={"arquivo": "acude"})
    display(df_2)

,data,lat,lon,precip_mm,municipio,estado,uf,regiao,arquivo
0,2006-01-01,-7.75,-41.25,5.292850,Jacobina do Piauí,Piauí,PI,Nordeste,precipitacao_2006_com_local
1,2006-01-01,-7.75,-40.75,10.205985,Simões,Piauí,PI,Nordeste,precipitacao_2006_com_local
2,2006-01-01,-7.75,-40.25,20.805733,Trindade,Pernambuco,PE,Nordeste,precipitacao_2006_com_local
3,2006-01-01,-7.75,-39.75,6.968030,Granito,Pernambuco,PE,Nordeste,precipitacao_2006_com_local
4,2006-01-01,-7.75,-39.25,0.812125,Cedro,Pernambuco,PE,Nordeste,precipitacao_2006_com_local
...,...,...,...,...,...,...,...,...,...
750316,2026-09-29,-2.75,-39.25,NaN,NaN,NaN,NaN,NaN,precipitacao_2026_com_local
750317,2026-09-29,-2.75,-38.75,NaN,NaN,NaN,NaN,NaN,precipitacao_2026_com_local
750318,2026-09-29,-2.75,-38.25,NaN,NaN,NaN,NaN,NaN,precipitacao_2026_com_local
750319,2026-09-29,-2.75,-37.75,NaN,NaN,NaN,NaN,NaN,precipitacao_2026_com_local


,DateTime,Volume (hm³),acude
0,2006-01-01 00:00:00,42.59,reserv_atalho
1,2006-01-02 00:00:00,42.59,reserv_atalho
2,2006-01-03 00:00:00,NaN,reserv_atalho
3,2006-01-04 00:00:00,NaN,reserv_atalho
4,2006-01-05 00:00:00,NaN,reserv_atalho
...,...,...,...
30306,2026-09-26 00:00:00,40.86,reserv_rosario
30307,2026-09-27 00:00:00,40.80,reserv_rosario
30308,2026-09-28 00:00:00,40.73,reserv_rosario
30309,2026-09-29 00:00:00,40.67,reserv_rosario


In [4]:
'''
pre processamento e filtragem mantendo as variáveis quantitativas
'''

# unindo e filtrando as coordenadas dentro do raio permitido (50km)
# copiei de rebeca tbm
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Distância real em km entre dois pontos (fórmula de Haversine).

    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2]) #converte graus para radianos
    dlat = lat2 - lat1 #faz a diferença
    dlon = lon2 - lon1 #faz a diferença
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2 #haversine formula
    c = 2 * np.arcsin(np.sqrt(a)) # distancia angular em radianos sobre a esfera
    return 6371 * c  # multiplica pelo raio médio da Terra em km

# coordenadas dos açudes e raio máximo e cria uma tabela com a localização de cada açude
coordenadas_acudes = pd.DataFrame({
    'acude': ['reserv_atalho', 'reserv_rosario', 'reserv_olho_dagua', 'reserv_quixabinha'],
    'lat_acude': [-8.75, -6.75, -6.25, -7.25],
    'lon_acude': [-38.25, -39.25, -36.25, -38.75]
})

RAIO_MAXIMO_KM = 50.0 #raio maximo, modificar aqui se necessário

# padronizar as datas em ambos
df_2['DateTime'] = pd.to_datetime(df_2['DateTime']).dt.date
df_1['data'] = pd.to_datetime(df_1['data']).dt.date

# Ponto de precipitação mais próximo de cada açude (dentro do raio)
pontos_precip_unicos = df_1[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
mapeamento_proximidade = []

for _, acude in coordenadas_acudes.iterrows():
    distancias_km = haversine_km(
        acude['lat_acude'], acude['lon_acude'],
        pontos_precip_unicos['lat'], pontos_precip_unicos['lon']
    )

    idx_mais_proximo = distancias_km.idxmin()
    menor_distancia = distancias_km.loc[idx_mais_proximo]

    if menor_distancia <= RAIO_MAXIMO_KM:
        mapeamento_proximidade.append({
            'acude': acude['acude'],
            'lat_acude': acude['lat_acude'],
            'lon_acude': acude['lon_acude'],
            'lat_precip_mais_proxima': pontos_precip_unicos.loc[idx_mais_proximo, 'lat'],
            'lon_precip_mais_proxima': pontos_precip_unicos.loc[idx_mais_proximo, 'lon'],
            'distancia_km': menor_distancia
        })
    else:
        print(f"o açude '{acude['acude']}' foi ignorado: ponto de chuva mais próximo "
              f"a {menor_distancia:.2f} km (limite: {RAIO_MAXIMO_KM} km).")

df_acudes_mapeados = pd.DataFrame(mapeamento_proximidade)

if df_acudes_mapeados.empty:
    raise ValueError("Nenhum açude atendeu ao critério de distância.")

df_2_geo = pd.merge(df_2, df_acudes_mapeados, on='acude', how='inner')

df_2['data_str'] = pd.to_datetime(df_2['DateTime']).dt.strftime('%Y-%m-%d')
df_1['data_str'] = pd.to_datetime(df_1['data']).dt.strftime('%Y-%m-%d')

df_unificado = pd.merge(
    df_2_geo,
    df_1,
    left_on=['data_str', 'lat_precip_mais_proxima', 'lon_precip_mais_proxima'],
    right_on=['data_str', 'lat', 'lon'],
    how='inner'
)

print(f"Aqui é o total de registros válidos dentro do raio de {RAIO_MAXIMO_KM} km: {len(df_unificado)}")

df_unificado["Data"] = pd.to_datetime(df_unificado["data_str"])
df_unificado = df_unificado.sort_values(["acude", "Data"]).reset_index(drop=True)

# criando as colunas exatas do grafo do teste 4
df_unificado["Mês"] = df_unificado["Data"].dt.month  # Quali nominal
df_unificado["Volume Anterior (v-1)"] = df_unificado.groupby("acude")[
    "Volume (hm³)"
].shift(1)  # Quanti

# renomeando colunas numéricas de interesse
df_unificado = df_unificado.rename(
    columns={"precip_mm": "Precipitação Recente", "Volume (hm³)": "Volume Atual (v)"}
)

# removendo NaNs gerados pelo shift(1)
df_testes = df_unificado.dropna(
    subset=["Precipitação Recente", "Volume Anterior (v-1)", "Volume Atual (v)"]
).copy()

display(df_testes.head())

o açude 'reserv_atalho' foi ignorado: ponto de chuva mais próximo a 111.19 km (limite: 50.0 km).
o açude 'reserv_olho_dagua' foi ignorado: ponto de chuva mais próximo a 110.53 km (limite: 50.0 km).
Aqui é o total de registros válidos dentro do raio de 50.0 km: 15158


,DateTime,Volume Atual (v),acude,data_str,lat_acude,lon_acude,lat_precip_mais_proxima,lon_precip_mais_proxima,distancia_km,data,...,lon,Precipitação Recente,municipio,estado,uf,regiao,arquivo,Data,Mês,Volume Anterior (v-1)
1,2006-01-02,7.58,reserv_quixabinha,2006-01-02,-7.25,-38.75,-7.25,-38.75,0.0,2006-01-02,...,-38.75,44.768280,Mauriti,Ceará,CE,Nordeste,precipitacao_2006_com_local,2006-01-02,1,7.58
2,2006-01-03,7.57,reserv_quixabinha,2006-01-03,-7.25,-38.75,-7.25,-38.75,0.0,2006-01-03,...,-38.75,18.526501,Mauriti,Ceará,CE,Nordeste,precipitacao_2006_com_local,2006-01-03,1,7.58
3,2006-01-04,7.57,reserv_quixabinha,2006-01-04,-7.25,-38.75,-7.25,-38.75,0.0,2006-01-04,...,-38.75,0.000000,Mauriti,Ceará,CE,Nordeste,precipitacao_2006_com_local,2006-01-04,1,7.57
4,2006-01-05,7.57,reserv_quixabinha,2006-01-05,-7.25,-38.75,-7.25,-38.75,0.0,2006-01-05,...,-38.75,0.000000,Mauriti,Ceará,CE,Nordeste,precipitacao_2006_com_local,2006-01-05,1,7.57
5,2006-01-06,7.57,reserv_quixabinha,2006-01-06,-7.25,-38.75,-7.25,-38.75,0.0,2006-01-06,...,-38.75,0.000000,Mauriti,Ceará,CE,Nordeste,precipitacao_2006_com_local,2006-01-06,1,7.57
